<a href="https://colab.research.google.com/github/rociocolungar-byte/pia7/blob/main/PIA_RNAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q transformers datasets accelerate scikit-learn pandas
!pip install -q transformers accelerate scikit-learn

In [12]:
from google.colab import files
uploaded = files.upload()

Saving train.tsv to train.tsv


In [14]:
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

In [15]:
train_df = pd.read_csv("train.tsv", sep="\t")
test_df = pd.read_csv("test.tsv", sep="\t")
print(train_df.head())
print(train_df.columns)

   Unnamed: 0     id                                             prompt  \
0           0   5464                                          NO-PROMPT   
1           1  30129  Estos podrían ser preguntas, categorías de inf...   
2           2  19553                    -¿Desea algo? -Póngame una caja   
3           3  13005                                          NO-PROMPT   
4           4  16919                                          NO-PROMPT   

                                                text      label     model  \
0  Entrada en vigor. La presente Directiva entrar...      human  NO-MODEL   
1  Preguntas: 1. ¿Cuáles son los principales argu...  generated         F   
2  ¿Desea algo? Póngame una caja de madera. ¿Qué ...  generated         E   
3  @victor28088 1665 Tweets no originales, que as...      human  NO-MODEL   
4  De pequeño Dios me dio a elegir entre tener un...      human  NO-MODEL   

   domain  
0   legal  
1    wiki  
2  tweets  
3  tweets  
4  tweets  
Index(['Unname

In [16]:
print("Columnas:")
print(train_df.columns.tolist())
print("\nEtiquetas:")
print(train_df["label"].value_counts())
print("\nValores únicos:")
print(train_df["label"].unique())


print("\nEjemplo de texto:")
print(train_df["text"].iloc[0])


Columnas:
['Unnamed: 0', 'id', 'prompt', 'text', 'label', 'model', 'domain']

Etiquetas:
label
generated    16275
human        15787
Name: count, dtype: int64

Valores únicos:
['human' 'generated']

Ejemplo de texto:
Entrada en vigor. La presente Directiva entrará en vigor el día de su publicación en el Diario Oficial de la Unión Europea. Artículo 8. Destinatarios. Los destinatarios de la presente Directiva son los Estados miembros. Hecho en Bruselas, el 26 de mayo de 2003.


In [17]:
print("Dimensiones de entrenamiento:", train_df.shape)
print("Dimensiones de prueba:", test_df.shape)
print("\nDatos faltantes:")
print(train_df.isnull().sum())

Dimensiones de entrenamiento: (32062, 7)
Dimensiones de prueba: (20129, 7)

Datos faltantes:
Unnamed: 0    0
id            0
prompt        0
text          0
label         0
model         0
domain        0
dtype: int64


In [18]:
train_df = train_df[["text", "label"]].dropna()
test_df = test_df[["text", "label"]].dropna()

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "DeepESP/gpt2-spanish"
tokenizer_gpt = AutoTokenizer.from_pretrained(MODEL_NAME)
model_gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gpt.to(device)
model_gpt.eval()
print("GPT-2 cargado correctamente")
print("Dispositivo:", device)

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/840k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/499k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  261MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: DeepESP/gpt2-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 cargado correctamente
Dispositivo: cuda


In [20]:
import torch.nn.functional as F
import math
def calcular_perplejidad(texto):
   entradas = tokenizer_gpt(
       texto,
       return_tensors="pt",
       truncation=True,
       max_length=512
   )
   input_ids = entradas["input_ids"].to(device)
   attention_mask = entradas["attention_mask"].to(device)
   with torch.no_grad():
       outputs = model_gpt(input_ids)
       logits = outputs.logits[:, :-1, :]
       targets = input_ids[:, 1:]
       perdida = F.cross_entropy(
           logits.reshape(-1, logits.size(-1)),
           targets.reshape(-1),
           reduction="mean"
       )
       perplejidad = torch.exp(perdida).item()
   return perplejidad

In [21]:
texto = "Hola, buenos días. ¿Cómo puedo ayudarte?"
resultado = calcular_perplejidad(texto)
print("Perplejidad:", resultado)

Perplejidad: 14.75


In [22]:
textos = [
   "Hola, buenos días. ¿Cómo puedo ayudarte?",
   "Ayer fui al supermercado y olvidé comprar algunas cosas.",
   "El profe Jandro siempre anda a la moda.",
   "No manches, se me olvidó completamente hacer la tarea jajaja."
]
for texto in textos:
   p = calcular_perplejidad(texto)
   print(f"Texto: {texto}")
   print(f"Perplejidad: {p:.2f}\n")

Texto: Hola, buenos días. ¿Cómo puedo ayudarte?
Perplejidad: 14.75

Texto: Ayer fui al supermercado y olvidé comprar algunas cosas.
Perplejidad: 70.12

Texto: El profe Jandro siempre anda a la moda.
Perplejidad: 352.00

Texto: No manches, se me olvidó completamente hacer la tarea jajaja.
Perplejidad: 133.50



In [23]:
print("Etiquetas:")
print(train_df["label"].value_counts())
print("\nValores únicos:")
print(train_df["label"].unique())

Etiquetas:
label
generated    16275
human        15787
Name: count, dtype: int64

Valores únicos:
['human' 'generated']


In [24]:
from sklearn.model_selection import train_test_split
train_part, val_part = train_test_split(
   train_df,
   test_size=0.10,
   random_state=42,
   stratify=train_df["label"]
)
print("Entrenamiento:", len(train_part))
print("Validación:", len(val_part))
print("Prueba:", len(test_df))

Entrenamiento: 28855
Validación: 3207
Prueba: 20129


In [25]:

mapa = {
   "human": 0,
   "generated": 1
}
train_part = train_part.copy()
val_part = val_part.copy()
test_df = test_df.copy()
for df in [train_part, val_part, test_df]:
   df["label"] = df["label"].map(mapa)
print(train_part["label"].value_counts())
print(val_part["label"].value_counts())
print(test_df["label"].value_counts())

label
1    14647
0    14208
Name: count, dtype: int64
label
1    1628
0    1579
Name: count, dtype: int64
label
1    11209
0     8920
Name: count, dtype: int64


In [26]:
import time
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
tokenizer_gpt.padding_side = "right"
def calcular_perplejidades(textos, batch_size=16):
   resultados = []
   for inicio in tqdm(range(0, len(textos), batch_size)):
       lote = textos[inicio:inicio + batch_size]
       entradas = tokenizer_gpt(
           lote,
           return_tensors="pt",
           padding=True,
           truncation=True,
           max_length=256
       )
       input_ids = entradas["input_ids"].to(device)
       attention_mask = entradas["attention_mask"].to(device)
       with torch.no_grad():
           logits = model_gpt(input_ids).logits
           logits = logits[:, :-1, :]
           targets = input_ids[:, 1:]
           mask = attention_mask[:, 1:]
           perdidas = F.cross_entropy(
               logits.reshape(-1, logits.size(-1)),
               targets.reshape(-1),
               reduction="none"
           ).reshape(targets.shape)
           perdidas = perdidas * mask
           promedio = perdidas.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
           ppl = torch.exp(promedio)
           resultados.extend(ppl.cpu().numpy())
   return np.array(resultados)

In [27]:
inicio = time.time()
ppl_val = calcular_perplejidades(
   val_part["text"].tolist()
)
tiempo_val = time.time() - inicio
# Prueba
inicio = time.time()
ppl_test = calcular_perplejidades(
   test_df["text"].tolist()
)
tiempo_test = time.time() - inicio
print("Tiempo de validación:", tiempo_val)
print("Tiempo de prueba:", tiempo_test)

  0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/1259 [00:00<?, ?it/s]

Tiempo de validación: 11.521889686584473
Tiempo de prueba: 66.80796933174133


In [28]:
from sklearn.metrics import f1_score
y_val = val_part["label"].values
umbrales = np.quantile(
   ppl_val,
   np.linspace(0.01, 0.99, 200)
)
mejor_f1 = 0
mejor_umbral = None
for umbral in umbrales:
   pred = (ppl_val <= umbral).astype(int)
   f1 = f1_score(
       y_val,
       pred,
       average="macro"
   )
   if f1 > mejor_f1:
       mejor_f1 = f1
       mejor_umbral = umbral
print("Mejor umbral:", mejor_umbral)
print("F1 de validación:", mejor_f1)

Mejor umbral: 47.34375
F1 de validación: 0.6257268265835965


In [29]:
from sklearn.metrics import (
   classification_report,
   confusion_matrix,
   f1_score,
   accuracy_score
)
y_test = test_df["label"].values
pred_test = (ppl_test <= mejor_umbral).astype(int)
print("F1 macro:", f1_score(
   y_test, pred_test, average="macro"
))
print("\nAccuracy:", accuracy_score(
   y_test, pred_test
))
print("\nReporte de clasificación:")
print(classification_report(
   y_test,
   pred_test,
   target_names=["Humano", "Máquina"]
))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

F1 macro: 0.711645193921855

Accuracy: 0.7131005017636246

Reporte de clasificación:
              precision    recall  f1-score   support

      Humano       0.62      0.88      0.73      8920
     Máquina       0.86      0.58      0.69     11209

    accuracy                           0.71     20129
   macro avg       0.74      0.73      0.71     20129
weighted avg       0.76      0.71      0.71     20129


Matriz de confusión:
[[7892 1028]
 [4747 6462]]
